In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from hurobo.utils.notebook import setup_notebook

ROOT = setup_notebook()

# Cleaning recorded clips before extraction

A raw phone clip has long stretches with no hand, sometimes a second hand, and often more than one grasp. `hurobo.clip_clean` cuts each clip down to **one clean stretch** and rejects clips that cannot give a demo:

**Frame filters** (a frame is unusable if any applies): no hand, more than one hand, bad hand fit (PnP reprojection error), hand touching the image edge, implausible depth.
**Cut** (the first gap-free run of usable frames that contains a full cycle): starts with the pinch open, closes once, opens again; everything before, after and in between is removed, with 2 s kept after the release. A pinch already closed on entry is trimmed, short open flickers inside a grasp are not releases.
**Clip filters:** the kept stretch must last 1.5 to 30 s and the board, bowl and plate markers must be visible in the first second.

The result is `cuts.json` next to the clips; `scripts/02_extract_demos.py --cuts` extracts only inside the cut (the table pose and the layout still come from the clip's first frames).

In [2]:
import pandas as pd

from hurobo.camera import CameraIntrinsics
from hurobo.clip_clean import ClipAnalyzer, ClipCleaner, find_clips
from hurobo.real_setup import DEFAULT_MEASUREMENTS, RealSetup
from hurobo.recording import RecordingSpec
from hurobo.utils.notebook import existing_paths

clip_paths = existing_paths(
    {path.name: path for path in find_clips(str(ROOT / "data/raw/e1/test*.mp4"))},
    "record test clips into data/raw/e1/",
)
intrinsics = CameraIntrinsics.load(ROOT / "data/camera_intrinsics_s10e.npz")
analyzer = ClipAnalyzer(
    intrinsics, RecordingSpec().estimator(intrinsics, RealSetup.load(DEFAULT_MEASUREMENTS))
)
cleaner = ClipCleaner()
signals = {name: analyzer.analyze_cached(path) for name, path in clip_paths.items()}
cuts = {name: cleaner.propose(clip_signals) for name, clip_signals in signals.items()}
cleaner.config

CleaningConfig(max_hands=1, max_reprojection_px=15.0, min_edge_margin_px=8.0, depth_range_m=(0.15, 3.0), max_gap_s=0.4, margin_after_release_s=2.0, min_closed_s=0.2, min_open_s=0.3, min_duration_s=1.5, max_duration_s=30.0, require_layout_markers=True)

One row per clip: how much of it is usable, and the proposed cut.

In [3]:
pd.DataFrame(
    {
        name: {
            "duration s": round(float(s.times_s[-1] - s.times_s[0]), 1),
            "hand fitted": f"{s.fitted.mean():.0%}",
            "kept": f"{cuts[name].start_s:.2f}-{cuts[name].end_s:.2f} s" if cuts[name].window_s else "-",
            "accepted": cuts[name].accepted,
            "why": "; ".join(cuts[name].reasons + cuts[name].notes),
        }
        for name, s in signals.items()
    }
).T

,duration s,hand fitted,kept,accepted,why
test_01.mp4,19.5,35%,-,False,no stretch with one hand where the pinch opens...
test_02.mp4,16.8,26%,-,False,no stretch with one hand where the pinch opens...
test.mp4,21.2,61%,2.70-12.60 s,True,cut 2.70 s at the start (no usable hand or pin...
test_04.mp4,26.2,67%,5.73-16.06 s,True,cut 5.73 s at the start (no usable hand or pin...


Each clip as a timeline: a red mark is a frame one filter rejects, orange is the pinch closed, the shaded parts are removed. Hover for the time. Use `CLIP` to look at another one.

In [4]:
from hurobo.utils.viewer import clip_cut_figure

CLIP = next(iter(signals))
clip_cut_figure(signals[CLIP], cleaner, cuts[CLIP]).show()

.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## A cleaned clip: `test`

The proposed cut, run through the extraction exactly as `scripts/02_extract_demos.py --cuts` does (the measured marker and board sizes are the defaults of `RecordingSpec` and `measurements.yaml`). `ClipReport` lists the gates the demo passed or failed.

In [5]:
from hurobo.recording import extract_board_clip

CLEAN = "test.mp4"
setup = RealSetup.load(DEFAULT_MEASUREMENTS)
cut = cuts[CLEAN]
demo, report = extract_board_clip(clip_paths[CLEAN], intrinsics, setup, window_s=cut.window_s)
print(
    f"kept {cut.start_s:.2f}-{cut.end_s:.2f} s of {signals[CLEAN].times_s[-1]:.1f} s; usable: {report.usable}"
)
print("gate failures:", report.reasons or "none")
print(
    f"hand in {report.hand_fraction:.0%} of the kept frames, anchor scale {report.anchor_scale:.2f}, "
    f"grasp {100 * report.grasp_to_bowl_m:.1f} cm from the bowl centre, "
    f"release {100 * report.release_to_plate_xy_m:.1f} cm from the plate marker"
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792177164.126259  115413 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792177164.135484  115413 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


W0000 00:00:1792177164.931605  115413 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


kept 2.70-12.60 s of 21.2 s; usable: True
gate failures: none
hand in 100% of the kept frames, anchor scale 1.00, grasp 8.1 cm from the bowl centre, release 7.3 cm from the plate marker


The hand path of the kept stretch in the table frame (blue free, red grasping):

In [6]:
from hurobo.utils.viewer import trajectory_figure

trajectory_figure(demo.tcp_pos_table, demo.is_grasping, title=f"{CLEAN}: TCP in the board frame (m)").show()

The diagnostic video of the whole clip: what was detected in every frame (bowl pad orange, plate marker magenta, board cyan, hand skeleton), the status panel, and a strip at the bottom with the detections of the whole clip. The yellow bar above the strip is the kept range; frames outside it are dimmed and labelled.

In [7]:
from hurobo.recording import diagnose_capture
from hurobo.utils.notebook import show_video

diagnosis = diagnose_capture(clip_paths[CLEAN], intrinsics, setup)
out_dir = ROOT / "results/clip_cleaning"
show_video(diagnosis.write_video(out_dir / "test_diagnosis.mp4", window_s=cut.window_s), width=720)

W0000 00:00:1792177177.535979  115710 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792177177.544350  115710 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


W0000 00:00:1792177216.753774  116500 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792177216.762556  116500 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


The clean clip: only the kept frames, the part that goes into extraction.

In [8]:
show_video(
    diagnosis.write_video(out_dir / "test_clean.mp4", window_s=cut.window_s, only_window=True), width=720
)

To adjust the cuts by hand, open the clip cutter (it shows the same lanes beside the video, with draggable start and end handles, and saves `cuts.json`):

```bash
uv run python scripts/21_clip_cutter.py data/raw/e1 --intrinsics data/camera_intrinsics_s10e.npz
```

In [9]:
from hurobo.recording import session_commands

for what, command in session_commands()[2:4]:
    print(f"# {what}\n{command}\n")

# 3. clean the clips: propose a cut per clip (one hand, one close-open cycle), then review them
uv run python scripts/20_clean_clips.py data/raw/e1 --setup data/raw/setup_01/measurements.yaml --intrinsics data/camera_intrinsics_s10e.npz
uv run python scripts/21_clip_cutter.py data/raw/e1 --setup data/raw/setup_01/measurements.yaml --intrinsics data/camera_intrinsics_s10e.npz

# 4. after recording: human clips to table-frame demos + QC report (inside the cuts)
uv run python scripts/02_extract_demos.py data/raw/e1/clip_*.mp4 --setup data/raw/setup_01/measurements.yaml --intrinsics data/camera_intrinsics_s10e.npz --cuts data/raw/e1/cuts.json --out data/e1/demos

